In [10]:
import pandas as pd
import json

plik = "dane.xlsx"

# Wczytujemy bez interpretowania nagłówków
df = pd.read_excel(plik, sheet_name="Data", header=None)

# DATA i CZAS są wspólne dla wszystkich stacji
daty = df.iloc[4:, 0]
czasy = df.iloc[4:, 1]

wynik = []

# Szukamy wszystkich kolumn PM25 w wierszu nagłówków
for col in range(df.shape[1]):

    naglowek = df.iloc[3, col]

    if pd.isna(naglowek):
        continue

    if not str(naglowek).endswith("_PM25"):
        continue

    # W Twoim pliku metadane są 5 kolumn wcześniej niż PM25.
    #
    # np.
    # C  = metadata/BATTERY
    # D  = NO2
    # E  = PA
    # F  = PM01
    # G  = PM10
    # H  = PM25
    metadata_col = col - 5

    adres = df.iloc[0, metadata_col]
    nazwa = df.iloc[1, metadata_col]
    lokalizacja = df.iloc[2, metadata_col]

    # wartości PM25
    pm25_values = df.iloc[4:, col]

    pomiary = []

    for data, czas, pm25 in zip(daty, czasy, pm25_values):

        # "--", puste pola itd. pomijamy
        if pd.isna(pm25) or str(pm25).strip() == "--":
            continue

        try:
            pm25 = float(str(pm25).replace(",", "."))
        except ValueError:
            continue

        # Data
        if isinstance(data, pd.Timestamp):
            data = data.strftime("%Y-%m-%d")
        else:
            data = str(data).split(" ")[0]

        # Czas
        czas = str(czas)

        pomiary.append({
            "data": data,
            "czas": czas,
            "wartosc": pm25
        })

    wynik.append({
        "adres": None if pd.isna(adres) else str(adres),
        "nazwa": None if pd.isna(nazwa) else str(nazwa),
        "lokalizacja": None if pd.isna(lokalizacja) else str(lokalizacja),
        "pm25": pomiary
    })


# zapis JSON
with open("pm25.json", "w", encoding="utf-8") as f:
    json.dump(
        wynik,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Liczba stacji:", len(wynik))

Liczba stacji: 109


In [12]:
# ============================================================
# PODSUMOWANIE DOPASOWANIA
# ============================================================

xlsx_stations = set(long_df["adres"].dropna().unique())
matched_stations = set(
    long_df.loc[
        long_df["station_id"].notna(),
        "adres"
    ].unique()
)

missing_stations = sorted(xlsx_stations - matched_stations)

print("\n=== PODSUMOWANIE ===")
print("Liczba stacji w XLSX:", len(xlsx_stations))
print("Połączono:", len(matched_stations))
print("Nie połączono:", len(missing_stations))

if missing_stations:
    print("\n=== STACJE BEZ DOPASOWANIA ===")
    for station in missing_stations:
        print(" -", station)
else:
    print("\nWszystkie stacje zostały poprawnie dopasowane.")

print("\n=== FORMAT WYNIKOWY ===")
print("Liczba pomiarów w czasie:", final.shape[0])
print("Liczba stacji w wyniku:", final.shape[1] - 1)
print("Rozmiar tabeli:", final.shape)

print("\nPierwsze wiersze:")
print(final.head())


=== PODSUMOWANIE ===
Liczba stacji w XLSX: 104
Połączono: 99
Nie połączono: 5

=== STACJE BEZ DOPASOWANIA ===
 - Bartnicza 8
 - Bulwary Karskiego
 - Koniecpolska 14
 - Nowoursynowska 210/212
 - Orłów Piastowskich 47

=== FORMAT WYNIKOWY ===
Liczba pomiarów w czasie: 744
Liczba stacji w wyniku: 99
Rozmiar tabeli: (744, 100)

Pierwsze wiersze:
                 data  1 Praskiego Pułku WP 116  Aleja Komandosów 8  \
0 2026-07-01 00:00:00                       4.7                 4.4   
1 2026-07-01 01:00:00                       3.9                 4.7   
2 2026-07-01 02:00:00                       3.7                 3.9   
3 2026-07-01 03:00:00                       3.4                 3.7   
4 2026-07-01 04:00:00                       3.4                 3.9   

   Aleja Krakowska 257  Arkuszowa 202  Armii Krajowej 39  Astronautów 17  \
0                  6.1            5.3                4.7             5.1   
1                  5.9            4.2                5.2             5.0   
